# Analyse de l'impact de l'IA et des réseaux sociaux sur la santé et les résultats scolaires des étudiants

## Contexte et Problème Métier

## Problématique

L'utilisation croissante des outils d'IA et des réseaux sociaux chez les étudiants soulève des questions majeures sur leur santé mentale, leur santé physique et leurs résultats scolaires.

Dans le cadre de notre projet capstone Data Science & Machine Learning, nous avons pris le plaisir de comprendre les relations existantes entre ces questions préoccupantes afin de mettre en place des politiques de prévention et d'accompagnement des apprenants.

## Objectifs du Projet

Notre objectif est de :
* Prédire le score de santé mentale via l'algorithme de régression
* Classer les étudiants à risque (santé mentale < 60) via une classification binaire
* Identifier les facteurs d'influence à travers l'interprétabilité

## Source des Données

* Dataset : AI_SocialMedia_Student_Dataset.csv
* Origine : Kaggle (AI and Social Media Impact on Student Health and Grades)
* Taille : 16 000 enregistrements, 10 variables
* Variables :
    - Student_ID : Identifiant unique
    - Age : Âge (13-25 ans)
    - Gender : Genre (Male, Female, Non-binary)
    - Education_Level : Niveau d'éducation (High School, College, University)
    - Daily_Social_Media_Hours : Heures quotidiennes sur les réseaux sociaux
    - Daily_AI_Tool_Usage_Hours : Heures quotidiennes d'utilisation d'outils IA
    - Sleep_Hours : Heures de sommeil
    - Physical_Activity_Hours : Heures d'activité physique
    - Mental_Health_Score : Score de santé mentale (0-100) [CIBLE]
    - Physical_Health_Score : Score de santé physique (0-100)

# **1. Exploration des Données (EDA)**
## 1.1 Chargement et Inspection Initiale

In [20]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
pip install matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
pip install seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
 pip install lightgbm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
pip install shap

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
pip install optuna

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuration
pd.set_option('display.width', 1400)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', '{:.2f}'.format)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
RANDOM_STATE = 42

# Chargement
df = pd.read_csv('AI_SocialMedia_Student_Dataset.csv')

print(f"Dimensions (lignes, colonnes) : {df.shape}")
print(f"Colonnes   : {df.columns.tolist()}")
print(f"\nTypes de données :")
print(df.dtypes)
print(f"\nValeurs manquantes :")
print(df.isnull().sum())
print(f"\nDoublons : {df.duplicated().sum()}")

Dimensions (lignes, colonnes) : (16000, 10)
Colonnes   : ['Student_ID', 'Age', 'Gender', 'Education_Level', 'Daily_Social_Media_Hours', 'Daily_AI_Tool_Usage_Hours', 'Sleep_Hours', 'Physical_Activity_Hours', 'Mental_Health_Score', 'Physical_Health_Score']

Types de données :
Student_ID                       str
Age                            int64
Gender                           str
Education_Level                  str
Daily_Social_Media_Hours     float64
Daily_AI_Tool_Usage_Hours    float64
Sleep_Hours                  float64
Physical_Activity_Hours      float64
Mental_Health_Score          float64
Physical_Health_Score        float64
dtype: object

Valeurs manquantes :
Student_ID                   0
Age                          0
Gender                       0
Education_Level              0
Daily_Social_Media_Hours     0
Daily_AI_Tool_Usage_Hours    0
Sleep_Hours                  0
Physical_Activity_Hours      0
Mental_Health_Score          0
Physical_Health_Score        0
dtype: in

## 1.2 Statistiques Descriptives

In [29]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML

# 1. Statistiques des variables numériques
stats_num = df.select_dtypes(include=np.number).describe().T
stats_num['skewness'] = df.select_dtypes(include=np.number).skew()
stats_num['kurtosis'] = df.select_dtypes(include=np.number).kurtosis()

# Style du tableau numérique
styled_stats = stats_num.style.background_gradient(cmap='Blues', subset=['mean', '50%'])\
    .format('{:.2f}')\
    .set_properties(**{
        'text-align': 'center',
        'border': '1px solid #dee2e6',
        'font-family': 'sans-serif'
    })\
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#1f77b4'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center')]},
        {'selector': 'tr:hover', 'props': [('background-color', '#f5f5f5')]}
    ])

# 2. Distribution des variables catégorielles sous forme de petits tableaux stylisés
cat_gender = df['Gender'].value_counts(normalize=True).to_frame(name='Proportion').round(3)
cat_edu = df['Education_Level'].value_counts(normalize=True).to_frame(name='Proportion').round(3)

styled_gender = cat_gender.style.background_gradient(cmap='Purples')\
    .format('{:.1%}')\
    .set_properties(**{'text-align': 'center', 'border': '1px solid #dee2e6', 'font-family': 'sans-serif'})\
    .set_table_styles([{'selector': 'th', 'props': [('background-color', '#6f42c1'), ('color', 'white'), ('font-weight', 'bold')]}])

styled_edu = cat_edu.style.background_gradient(cmap='GnBu')\
    .format('{:.1%}')\
    .set_properties(**{'text-align': 'center', 'border': '1px solid #dee2e6', 'font-family': 'sans-serif'})\
    .set_table_styles([{'selector': 'th', 'props': [('background-color', '#20c997'), ('color', 'white'), ('font-weight', 'bold')]}])


# Affichage structuré et esthétique
display(HTML("""
<div style='font-family: sans-serif;'>
    <h2 style='color: #1f77b4; border-bottom: 2px solid #1f77b4; padding-bottom: 8px;'>📊 Tableau Harmonique des Statistiques Descriptives</h2>
    <p style='color: #555;'>Statistiques de tendance centrale, de dispersion et de forme (asymétrie et aplatissement) pour les variables quantitatives.</p>
</div>
"""))
display(styled_stats)

display(HTML("""
<br>
<div style='font-family: sans-serif;'>
    <h3 style='color: #495057; border-bottom: 1.5px solid #6c757d; padding-bottom: 5px;'>✨ Distribution des Variables Catégorielles</h3>
</div>
<div style='display: flex; gap: 50px; font-family: sans-serif; margin-top: 15px;'>
    <div style='flex: 1;'>
        <h4 style='color: #6f42c1;'>👥 Répartition par Genre</h4>
        """ + styled_gender.to_html() + """
    </div>
    <div style='flex: 1;'>
        <h4 style='color: #20c997;'>🎓 Répartition par Niveau d'Éducation</h4>
        """ + styled_edu.to_html() + """
    </div>
</div>
<br>
"""))

AttributeError: The '.style' accessor requires jinja2

## **Interprétation des Statistiques Descriptives**

À la lumière des résultats numériques et des distributions catégorielles, plusieurs observations majeures peuvent être formulées :



1. **Aucune valeur manquante**, donc pas d'imputation nécessaire


2. **Profil Démographique de la Cohorte :**
   - **Âge :** La population étudiée est jeune, avec une moyenne d'âge de **19 ans** (allant de 13 à 25 ans) et une distribution uniforme (kurtosis négatif de -1.22).
   - **Genre :** La cohorte est approximativement équilibrée entre les **hommes (48%)** et les **femmes (48%)**, avec une minorité représentée par les personnes non-binaires (4%).
   - **Éducation :** Les étudiants proviennent principalement du secondaire (High School : 38%), suivis par l'université (University : 31%) et le collège (College : 31%).

3. **Hygiène de Vie et Vie Numérique :**
   - **Temps d'Écran Social (Daily_Social_Media_Hours) :** Les étudiants passent en moyenne **4,54 heures par jour** sur les réseaux sociaux. La valeur maximale atteint un extrême de 14 heures.
   - **Usage des outils d'IA (Daily_AI_Tool_Usage_Hours) :** L'usage moyen de l'IA est plus modéré, s'élevant à **2,59 heures quotidiennes** et au maximum 9 heures par jours. On a une distribution légèrement asymétrique à droite (skew=0.41).
   - **Sommeil et Activité Physique :** Les étudiants dorment en moyenne **6,55 heures** par nuit, ce qui est légèrement en dessous des recommandations de santé publique (**8 heures** minimum). L'activité physique moyenne est très faible (**1,25 heure par jour**).

4. **Santé et Bien-être :**
   - **Santé Physique (Physical_Health_Score) :** Le score de santé physique moyen est très satisfaisant (**88.02/100**), avec une asymétrie vers la gauche (skewness de -0.75). Cela indique que la majorité des étudiants se sentent en bonne santé physique.
   - **Santé Mentale (Mental_Health_Score, la Cible) :** Le score de santé mentale moyen s'établit à **72.49/100**. Globalement, on peut en déduire qu'en moyenne,les étudiantes ont un niveau de santé mentale plutôt bon et stable (car proche de 72,5 sur une échelle de 0 à 100). Cependant, l'asymétrie négative (**Skewness = -0.68**) indique la présence d'une queue de distribution vers les scores les plus bas. Le coefficient d'asymétrie étant négatif, cela signifie que la distribution n'est pas parfaitement symétrique (en cloche) mais qu'elle est étirée vers la gauche.
   On peut donc dire que la grande majorité des étudiants ont des scores individuels supérieurs à la moyenne générale (la médiane est plus élevée que la moyenne), mais qu'il existe un sous-groupe d'étudiants qui présentent des scores particulièrement bas. Ce sont ces valeurs anormalement basses (la "queue" à gauche de la distribution) qui tirent la moyenne globale vers le bas. Ainsi, nous avons un sous-groupe d'étudiants en détresse psychologique ou "à risque" que nous identifierons par la suite.

# 1.3 Visualisations Pertinentes
### Visualisation 1 : Distribution de la cible (santé mentale) et des variables clés

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Distribution de la cible
axes[0, 0].hist(df['Mental_Health_Score'], bins=40, color='steelblue', edgecolor='white')
axes[0, 0].axvline(df['Mental_Health_Score'].mean(), color='red', linestyle='--',
                    label=f'Moyenne = {df["Mental_Health_Score"].mean():.1f}')
axes[0, 0].axvline(60, color='orange', linestyle='--', label='Seuil risque = 60')
axes[0, 0].set_title('Distribution du Score de Santé Mentale', fontsize=12)
axes[0, 0].legend()
axes[0, 0].set_xlabel('Score de santé mentale')

# Heures réseaux sociaux
axes[0, 1].hist(df['Daily_Social_Media_Hours'], bins=40, color='coral', edgecolor='white')
axes[0, 1].set_title('Heures quotidiennes sur les réseaux sociaux', fontsize=12)
axes[0, 1].set_xlabel('Heures')

# Heures IA
axes[0, 2].hist(df['Daily_AI_Tool_Usage_Hours'], bins=40, color='mediumseagreen', edgecolor='white')
axes[0, 2].set_title('Heures quotidiennes d\'utilisation d\'IA', fontsize=12)
axes[0, 2].set_xlabel('Heures')

# Heures de sommeil
axes[1, 0].hist(df['Sleep_Hours'], bins=40, color='mediumpurple', edgecolor='white')
axes[1, 0].set_title('Heures de sommeil', fontsize=12)
axes[1, 0].set_xlabel('Heures')

# Activité physique
axes[1, 1].hist(df['Physical_Activity_Hours'], bins=40, color='darkorange', edgecolor='white')
axes[1, 1].set_title('Heures d\'activité physique', fontsize=12)
axes[1, 1].set_xlabel('Heures')

# Score de santé physique
axes[1, 2].hist(df['Physical_Health_Score'], bins=40, color='teal', edgecolor='white')
axes[1, 2].set_title('Score de Santé Physique', fontsize=12)
axes[1, 2].set_xlabel('Score')

plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

# **Interprétation des graphiques**

### **📊 Analyse Détaillée des Distributions des Variables Clés**

À la lumière des visualisations générées dans la première figure, nous pouvons analyser en profondeur les caractéristiques de distribution de chaque facteur :

#### **1. Le Score de Santé Mentale (`Mental_Health_Score` — Cible)**
* **Forme de la distribution :** Légèrement asymétrique vers la gauche (asymétrie négative ou *skewness* de $-0,68$). La majorité des étudiants ont des scores groupés entre $65$ et $85$ (médiane à $73,75$ et moyenne à $72,49$).
* **Seuil de risque critique ($< 60$) :** On observe visuellement une longue traîne s'étendant vers la gauche. La ligne orange représentant le **Seuil de risque à 60** isole environ **$10,4\%$** des étudiants. Ce sous-groupe présente des scores anormalement bas, justifiant l'utilisation d'un classifieur binaire pour cibler cette population en détresse.

#### **2. L'Exposition Numérique Quotidienne**
* **Réseaux Sociaux (`Daily_Social_Media_Hours`) :** La distribution est très étalée et relativement plate (kurtosis négatif de $-0,35$), avec une moyenne élevée de **$4,54$ heures/jour**. On remarque un groupe non négligeable d'utilisateurs extrêmes dépassant les $8$ heures et allant jusqu'à $14$ heures quotidiennes, représentant des profils à risque d'addiction numérique.
* **Outils d'IA (`Daily_AI_Tool_Usage_Hours`) :** Contrairement aux réseaux sociaux, la distribution est plus resserrée et asymétrique à droite (*skewness* de $+0,41$), centrée autour d'une moyenne modérée de **$2,59$ heures/jour**. L'utilisation intensive reste rare (très peu d'étudiants dépassent les $6$ heures), ce qui suggère un usage plus utilitaire ou académique et potentiellement moins addictif.

#### **3. Les Habitudes de Vie (Sommeil et Sport)**
* **Sommeil (`Sleep_Hours`) :** La courbe est centrée sur **$6,55$ heures/nuit**, révélant un déficit de sommeil chronique généralisé puisque la distribution culmine bien en dessous du seuil recommandé de $8$ heures par nuit.
* **Activité Physique (`Physical_Activity_Hours`) :** C'est la distribution la plus déséquilibrée avec une forte asymétrie positive (*skewness* de $+0,62$). On constate un pic massif proche de $0$, indiquant qu'une part écrasante d'étudiants pratique très peu, voire aucune activité physique au quotidien.
* **Santé Physique (`Physical_Health_Score`) :** Malgré le manque de sport et de sommeil, la santé physique autodéclarée reste excellente pour la majorité des étudiants, culminant près de **$88/100$** avec une asymétrie vers la gauche. Cela s'explique par la jeunesse relative de la cohorte (moyenne d'âge de $19$ ans) dont l'organisme compense encore temporairement ces lacunes d'hygiène de vie.

Visualisation 2 : Matrice de Corrélation

In [ ]:
# Corrélations avec la cible

# Matrice de corrélation
cols_num = ['Age', 'Daily_Social_Media_Hours', 'Daily_AI_Tool_Usage_Hours',
            'Sleep_Hours', 'Physical_Activity_Hours',
            'Mental_Health_Score', 'Physical_Health_Score']

corr_matrix = df[cols_num].corr()
corr_target = corr_matrix['Mental_Health_Score'].drop('Mental_Health_Score').sort_values()

print("\nCorrélations avec Mental_Health_Score :")
print(corr_target.round(4))
print("---------------------------------------------------\n\n")


plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
plt.title('Matrice de Corrélation — Variables Numériques', fontsize=14)
plt.tight_layout()
plt.savefig('eda_correlation.png', dpi=120, bbox_inches='tight')
plt.show()

### **Interprétation des Corrélations**

En analysant les corrélations linéaires avec notre cible, le **Mental_Health_Score** (score de santé mentale), nous pouvons dégager plusieurs conclusions majeures :

1. **L'impact prédominant du temps d'écran :**
   - **Daily_Social_Media_Hours (r = -0.40) :** Il s'agit de la corrélation négative la plus forte parmi les variables initiales. Plus le temps passé quotidiennement sur les réseaux sociaux est élevé, plus le score de santé mentale a tendance à baisser.
   - **Screen_Sleep_Ratio (r = -0.43) & Total_Screen_Time (r = -0.39) :** Les variables créées confirment cette tendance. Le ratio entre le temps d'écran total et le sommeil présente la corrélation négative la plus forte de tout le jeu de données ($r = -0.43$), suggérant qu'un déséquilibre fort entre vie numérique et repos est un facteur de risque majeur.

2. **Le rôle protecteur du sommeil et de l'activité physique :**
   - **Sleep_Hours (r = +0.31) :** Un temps de sommeil plus long est positivement et significativement associé à une meilleure santé mentale.
   - **Physical_Activity_Hours (r = +0.25) & Physical_Health_Score (r = +0.25) :** L'activité physique et la santé physique globale agissent également comme des facteurs protecteurs notables.

3. **L'impact plus nuancé de l'IA :**
   - **Daily_AI_Tool_Usage_Hours (r = -0.11) :** L'utilisation des outils d'IA montre une corrélation négative avec le score de santé mentale, mais elle est beaucoup plus faible que celle des réseaux sociaux. L'usage de l'IA semble moins nocif en soi, ou son impact négatif est plus indirect.

4. **Absence d'effet de l'âge :**
   - **Age (r = 0.01) :** L'âge n'affiche aucune corrélation linéaire avec la santé mentale dans cette tranche d'âge de 13 à 25 ans.

## Visualisation 3 : Relations non-linéaires (pairplot simplifié)

In [ ]:
# Pairplot des variables les plus corrélées avec la cible
cols_plot = ['Mental_Health_Score', 'Daily_Social_Media_Hours',
             'Sleep_Hours', 'Physical_Health_Score']

sample = df[cols_plot].sample(2000, random_state=RANDOM_STATE)
g = sns.pairplot(sample, diag_kind='kde', plot_kws={'alpha': 0.3, 's': 10})
g.fig.suptitle('Relations entre les variables clés', y=1.02, fontsize=14)
plt.savefig('eda_pairplot.png', dpi=100, bbox_inches='tight')
plt.show()

### **📊 Analyse et Interprétation du Pairplot (Relations Non-Linéaires)**

L'examen des graphiques croisés (*pairplot*) construits sur un échantillon représentatif de $2\ 000$ étudiants permet d'affiner notre compréhension des dynamiques complexes qui régissent la santé mentale, au-delà de simples coefficients de corrélation linéaire :

#### **1. Analyse des Diagonales (Estimations de Densité par Noyau - KDE)**
* **`Mental_Health_Score` :** La courbe de densité confirme l'asymétrie négative observée précédemment. Elle montre que le bien-être psychologique ne suit pas une distribution strictement normale, mais possède un flanc gauche étiré, témoignant d'une population vulnérable bien distincte du profil de l'étudiant moyen.
* **`Daily_Social_Media_Hours` :** La forme de la courbe est étalée et presque uniforme sur une large plage, confirmant que la forte exposition numérique (entre $3$ et $7$ heures) touche une immense partie des étudiants de manière homogène.
* **`Sleep_Hours` :** On observe une courbe de sommeil resserrée avec un pic de densité prononcé autour de $6,5$ heures, illustrant à quel point la privation modérée de sommeil est devenue une norme comportementale chez ces jeunes.

#### **2. Analyse des Croisements (Nuages de Points & Tendances Non-Linéaires)**
* **Réseaux Sociaux vs Santé Mentale (`Daily_Social_Media_Hours` × `Mental_Health_Score`) :**
  * On n'observe pas une chute linéaire abrupte et uniforme dès la première heure d'utilisation.
  * Au contraire, le nuage de points révèle un **effet de seuil** : pour des usages modérés (en dessous de $3$ à $4$ heures), l'impact sur la santé mentale reste diffus et stable. Cependant, dès que l'on bascule dans l'utilisation intensive (au-delà de $6$ heures quotidiennes), la densité des scores élevés s'effondre drastiquement au profit d'une accumulation de points dans les bas scores.
* **Sommeil vs Santé Mentale (`Sleep_Hours` × `Mental_Health_Score`) :**
  * Il existe une relation positive visible, mais elle présente également une saturation non-linéaire.
  * Dormir très peu (moins de $5$ heures) s'associe systématiquement à une forte dispersion et à une chute sévère des scores de santé mentale.
  * En revanche, l'augmentation des heures de sommeil au-delà de $8$ heures montre des rendements marginaux décroissants (le score de santé mentale se stabilise et stagne), indiquant qu'il existe une plage optimale de sommeil protectrice.
* **Santé Physique vs Santé Mentale (`Physical_Health_Score` × `Mental_Health_Score`) :**
  * Le croisement montre une zone de forte densité indiquant une synergie positive globale (les étudiants se sentant bien physiquement ont tendance à déclarer une meilleure santé mentale).
  * Cependant, la présence de points dispersés dans les quadrants opposés rappelle que de bons indicateurs physiques n'excluent pas des situations de détresse psychologique isolées, souvent induites par d'autres facteurs comme la surcharge numérique.

Ces observations non-linéaires et ces effets de seuil renforcent l'intérêt d'utiliser des algorithmes basés sur des arbres de décision (comme **Random Forest** et **LightGBM**), qui excellent naturellement à segmenter l'espace des données selon ces paliers critiques contrairement aux modèles linéaires classiques.

# **2. Pipeline de Preprocessing et Feature Engineering**
## 2.1 Création de la Variable Cible (Classification)

In [ ]:
# Variable cible binaire : étudiant à risque (santé mentale < 60)
df['at_risk'] = (df['Mental_Health_Score'] < 60).astype(int)

# print(f"Distribution de la variable 'at_risk' :")
# print(df['at_risk'].value_counts())
# print(f"\nProportion de positifs : {df['at_risk'].mean():.3f} ({df['at_risk'].mean()*100:.1f}%)")

# Configuration esthétique
colors = ['#4ea8de', '#ff6b6b']
labels = ['Stable (Score >= 60)', 'À risque (Score < 60)']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 1. Graphique à barres (Countplot)
sns.countplot(x='at_risk', data=df, ax=axes[0], palette=colors, edgecolor='black', linewidth=1.2)
axes[0].set_title("Nombre d'étudiants par catégorie de risque", fontsize=14, fontweight='bold', pad=15)
axes[0].set_xlabel("Statut de l'étudiant", fontsize=12)
axes[0].set_ylabel("Nombre d'étudiants", fontsize=12)
axes[0].set_xticklabels(labels, fontsize=11)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 10), textcoords='offset points', fontsize=11, fontweight='bold')

# 2. Graphique circulaire (Pie Chart)
counts = df['at_risk'].value_counts()
axes[1].pie(counts, labels=labels, autopct='%1.1f%%', startangle=140, colors=colors,
           explode=(0, 0.1), shadow=True, textprops={'fontsize': 12, 'fontweight': 'bold'})
axes[1].set_title("Proportion d'étudiants à risque", fontsize=14, fontweight='bold', pad=15)

plt.suptitle("🎯 Analyse de la Variable Cible : Étudiants à Risque Mental", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 2.2 Feature Engineering

In [ ]:
# ==========================================
# FEATURE ENGINEERING & ENRICHISSEMENT DU DATASET
# ==========================================

# 1. Total_Screen_Time : Cumul du temps quotidien d'exposition aux écrans (Réseaux Sociaux + Outils IA)
df['Total_Screen_Time'] = df['Daily_Social_Media_Hours'] + df['Daily_AI_Tool_Usage_Hours']

# 2. Digital_Balance : Écart entre le temps récréatif (réseaux sociaux) et productif/utilitaire (IA)
df['Digital_Balance'] = df['Daily_Social_Media_Hours'] - df['Daily_AI_Tool_Usage_Hours']

# 3. Sleep_Deficit : Écart par rapport au seuil recommandé de 8h de sommeil (clippé à 0 pour éviter les valeurs négatives)
df['Sleep_Deficit'] = (8 - df['Sleep_Hours']).clip(lower=0)

# 4. Activity_Ratio : Ratio d'activité physique par rapport aux heures de repos (sommeil + 1h de marge pour éviter la division par zéro)
df['Activity_Ratio'] = df['Physical_Activity_Hours'] / (df['Sleep_Hours'] + 1)

# 5. Screen_Sleep_Ratio : Ratio mesurant le déséquilibre entre le temps d'écran total et le repos de l'étudiant
df['Screen_Sleep_Ratio'] = df['Total_Screen_Time'] / (df['Sleep_Hours'] + 0.1)

# 6. Age_Group : Discrétisation de l'âge en tranches démographiques cohérentes (Ados, Jeunes Adultes, etc.)
df['Age_Group'] = pd.cut(df['Age'], bins=[12, 15, 18, 21, 26], labels=['Jeune_Ado', 'Ado', 'Jeune_Adulte', 'Adulte'])

# 7. Heavy_Social_Media : Variable binaire identifiant les utilisateurs intensifs de réseaux sociaux (plus de 6 heures par jour)
df['Heavy_Social_Media'] = (df['Daily_Social_Media_Hours'] > 6).astype(int)

# 8. Sleep_Deprived : Variable binaire marquant le manque sévère de sommeil (moins de 6 heures par nuit)
df['Sleep_Deprived'] = (df['Sleep_Hours'] < 6).astype(int)

# ==========================================
# AFFICHAGE ESTHÉTIQUE DES RÉSULTATS
# ==========================================
from IPython.display import display, HTML

# Calcul de la corrélation des nouvelles variables créées avec notre cible
new_features = ['Total_Screen_Time', 'Sleep_Deficit', 'Screen_Sleep_Ratio', 'Heavy_Social_Media', 'Sleep_Deprived', 'Activity_Ratio']
correlations = df[new_features + ['Mental_Health_Score']].corr()['Mental_Health_Score'].drop('Mental_Health_Score').to_frame(name='Corrélation avec Mental_Health_Score').sort_values(by='Corrélation avec Mental_Health_Score')

# Mise en forme conditionnelle et design du tableau (Correction : suppression de l'argument 'center' non supporté par Styler.background_gradient)
styled_corr = correlations.style.background_gradient(cmap='RdYlGn', vmin=-0.5, vmax=0.5, subset=['Corrélation avec Mental_Health_Score'])\
    .format('{:.3f}')\
    .set_properties(**{
        'text-align': 'center',
        'border': '1px solid #dee2e6',
        'font-family': 'sans-serif',
        'padding': '10px'
    })\
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#343a40'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('font-family', 'sans-serif')]},
        {'selector': 'tr:hover', 'props': [('background-color', '#f8f9fa')]}
    ])

# Affichage propre
display(HTML("""
<div style='font-family: sans-serif; margin-bottom: 15px;'>
    <h3 style='color: #2c3e50; border-bottom: 2px solid #2c3e50; padding-bottom: 8px;'>⚙️ Feature Engineering & Analyse d'Impact</h3>
    <p style='color: #7f8c8d; font-size: 14px;'>Les variables dérivées ont été créées avec succès pour capturer les habitudes numériques et l'hygiène de vie des étudiants. Voici leurs relations linéaires avec le score de santé mentale :</p>
</div>
"""))
display(styled_corr)


# **Interprétation des résultats**

À la lumière des coefficients de corrélation obtenus pour nos nouvelles variables créées, voici l'analyse et l'interprétation de leur impact sur le score de santé mentale (Mental_Health_Score) :

1. Le déséquilibre majeur : Temps d'écran vs Sommeil
Screen_Sleep_Ratio (r = -0.43) : C'est la corrélation négative la plus forte de tout notre dataset. Cela démontre de manière flagrante que ce n'est pas uniquement le temps d'écran en soi qui nuit à la santé mentale, mais le déséquilibre qu'il crée par rapport au temps de récupération (sommeil). Plus ce ratio est élevé (c'est-à-dire beaucoup d'écrans pour très peu de sommeil), plus le score de santé mentale s'effondre.
2. L'exposition globale aux technologies
Total_Screen_Time (r = -0.39) : Sans surprise, le cumul du temps passé sur les réseaux sociaux et sur les outils d'IA est fortement corrélé négativement avec la santé mentale. L'omniprésence du numérique au quotidien reste un facteur de tension psychologique majeur.
3. Les comportements à risque extrême
Heavy_Social_Media (r = -0.32) : Le fait de dépasser le seuil critique de 6 heures par jour sur les réseaux sociaux est un indicateur de risque très net.
Sleep_Deficit (r = -0.31) et Sleep_Deprived (r = -0.25) : Le manque cumulé de sommeil (écart par rapport aux 8 heures recommandées) et le fait de dormir moins de 6 heures par nuit sont fortement préjudiciables. Cela confirme l'importance cruciale d'un sommeil réparateur sur la régulation de l'humeur et le bien-être cognitif.
4. Le facteur protecteur
Activity_Ratio (r = +0.18) : C'est la seule nouvelle variable de notre feature engineering à présenter une corrélation positive. Plus le ratio d'activité physique par rapport au temps de repos est élevé, meilleur est le score de santé mentale. Le sport agit ainsi comme un véritable bouclier thérapeutique et un régulateur de stress face à l'exposition numérique.

## 2.3 Construction du Pipeline Scikit-Learn

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold

# =========================================================================
# 1. DÉFINITION ET SÉLECTION DES FEATURES
# =========================================================================
# Cibles pour les deux tâches (Régression & Classification)
target_reg = 'Mental_Health_Score'
target_clf = 'at_risk'

# Suppression des identifiants (sans valeur prédictive), des cibles et de la variable corrélée
cols_to_drop = ['Student_ID', target_reg, target_clf, 'Age_Group']
feature_cols = [c for c in df.columns if c not in cols_to_drop]

# Séparation automatique entre variables quantitatives et qualitatives
cols_num = df[feature_cols].select_dtypes(include=np.number).columns.tolist()
cols_cat = df[feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Features numériques : {len(cols_num)}")
print(f"Features catégorielles : {len(cols_cat)}")
print(f"\nCols catégorielles : {cols_cat}")

# =========================================================================
# 2. CONSTITUTION DES PIPELINES DE PREPROCESSING (Scikit-Learn)
# =========================================================================
def construire_preprocesseur(cols_num, cols_cat):
    """
    Pipeline de preprocessing complet pour éviter le data leakage :
    - Pour les variables numériques : Imputation par la médiane + Standardisation (Centrage-Réduction)
    - Pour les variables catégorielles : Imputation par la valeur la plus fréquente + Encodage One-Hot
    """
    # Pipeline Numérique : Gère les données manquantes et ramène les variables à la même échelle
    prep_num = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    # Pipeline Catégoriel : Gère les données textuelles et les transforme en format exploitable (One-Hot)
    prep_cat = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    # Fusion des traitements par colonnes respectives via ColumnTransformer
    return ColumnTransformer([
        ('num', prep_num, cols_num),
        ('cat', prep_cat, cols_cat)
    ])

preprocesseur = construire_preprocesseur(cols_num, cols_cat)

# =========================================================================
# 3. SPLIT STRATIFIÉ EN ENSEMBLES D'ENTRAÎNEMENT ET DE TEST
# =========================================================================
X = df[feature_cols]
y_reg = df[target_reg]
y_clf = df[target_clf]

# Division 80/20 avec stratification sur 'at_risk' pour maintenir la même proportion
# d'étudiants en détresse dans les deux partitions
X_train, X_test, y_train_reg, y_test_reg, y_train_clf, y_test_clf = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=RANDOM_STATE, stratify=y_clf
)

print(f"\nTrain : {X_train.shape} | Test : {X_test.shape}")
print(f"Proportion positifs train : {y_train_clf.mean():.3f}")
print(f"Proportion positifs test  : {y_test_clf.mean():.3f}")

# **3. Modélisation**

## 3.1 Tâche 1 : Régression (Prédire le Score de Santé Mentale)


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import lightgbm as lgb

# =========================================================================
# 1. FONCTION D'ÉVALUATION DES MODÈLES DE RÉGRESSION
# =========================================================================
def evaluer_regression(model, X_train, y_train, X_test, y_test, nom):
    """
    Entraîne le modèle fourni et évalue ses performances sur le jeu de test.
    Calcule et affiche :
      - RMSE (Root Mean Squared Error) : pénalise plus fortement les grandes erreurs.
      - MAE (Mean Absolute Error) : erreur moyenne absolue en valeur réelle du score.
      - R² (Coefficient de détermination) : pourcentage de la variance expliquée par le modèle.
    """
    # Entraînement du modèle/pipeline sur les données d'apprentissage
    model.fit(X_train, y_train)

    # Prédiction sur les données de test non vues
    y_pred = model.predict(X_test)

    # Calcul des métriques de performance
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    # Affichage formaté des résultats
    print(f"{nom:<30} RMSE={rmse:.3f}  MAE={mae:.3f}  R²={r2:.4f}")

    # Retour du modèle entraîné et du dictionnaire de métriques pour archivage
    return model, {'rmse': rmse, 'mae': mae, 'r2': r2}

print("=" * 70)
print("RÉGRESSION — Prédiction du Mental_Health_Score")
print("=" * 70)

# Dictionnaire destiné à sauvegarder les scores de chaque modèle pour comparaison ultérieure
resultats_reg = {}

# =========================================================================
# 2. MODÈLE BASÉ SUR LA RÉGRESSION RIDGE (Baseline)
# =========================================================================
# La régression Ridge est un modèle linéaire avec régularisation L2 pour éviter le surapprentissage.
pipe_ridge = Pipeline([('prep', preprocesseur), ('model', Ridge(alpha=1.0))])
_, resultats_reg['Ridge'] = evaluer_regression(
    pipe_ridge, X_train, y_train_reg, X_test, y_test_reg, 'Ridge (baseline)')

# =========================================================================
# 3. MODÈLE BASÉ SUR LE RANDOM FOREST REGRESSOR
# =========================================================================
# Forêt d'arbres décisionnels. Robuste aux relations non-linéaires.
# n_estimators=200 : entraîne 200 arbres distincts.
# max_depth=12 : limite la profondeur de chaque arbre pour éviter le surapprentissage.
# n_jobs=-1 : utilise tous les cœurs du processeur pour accélérer l'entraînement.
pipe_rf = Pipeline([('prep', preprocesseur),
                     ('model', RandomForestRegressor(n_estimators=200, max_depth=12,
                                                      n_jobs=-1, random_state=RANDOM_STATE))])
_, resultats_reg['RandomForest'] = evaluer_regression(
    pipe_rf, X_train, y_train_reg, X_test, y_test_reg, 'Random Forest')

# =========================================================================
# 4. MODÈLE BASÉ SUR LIGHTGBM (Gradient Boosting)
# =========================================================================
# Algorithme basé sur le boosting d'arbres de décision, très rapide et souvent le plus précis.
# n_estimators=500 : nombre maximal d'itérations de boosting.
# learning_rate=0.05 : vitesse d'apprentissage (pas d'apprentissage modéré pour une convergence stable).
# num_leaves=31 : nombre maximum de feuilles par arbre.
pipe_lgbm = Pipeline([('prep', preprocesseur),
                       ('model', lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05,
                                                    num_leaves=31, n_jobs=-1,
                                                    random_state=RANDOM_STATE, verbose=-1))])
_, resultats_reg['LightGBM'] = evaluer_regression(
    pipe_lgbm, X_train, y_train_reg, X_test, y_test_reg, 'LightGBM')

### **3.1.1 Interprétation des Résultats de Régression**

Voici l'analyse comparative des performances de nos trois modèles pour la prédiction du score de santé mentale (`Mental_Health_Score`) :

| Modèle | RMSE | MAE | $R^2$ (Variance Expliquée) |
| :--- | :---: | :---: | :---: |
| **Random Forest** | **7.898** | **6.195** | **28.41%** |
| **Ridge (Baseline)** | 7.999 | 6.303 | 26.57% |
| **LightGBM** | 8.013 | 6.267 | 26.29% |

#### **Observations Clés :**

1. **Le Meilleur Modèle : Random Forest**
   * Le modèle **Random Forest** obtient les meilleures performances globales avec le **RMSE le plus faible (7.898)** et le **$R^2$ le plus élevé (28.41%)**.
   * Cela signifie qu'il parvient mieux à capturer les interactions non-linéaires complexes entre nos variables explicatives (comme les ratios temps d'écran/sommeil) et le score de santé mentale.

2. **Signification des Métriques d'Erreur :**
   * **MAE de 6.195** (pour le Random Forest) : En moyenne, nos prédictions du score de santé mentale s'écartent de **6.2 points** par rapport au score réel de l'étudiant (sur une échelle de 0 à 100). C'est une précision tout à fait honorable pour des données comportementales humaines.
   * **RMSE de 7.898** : Légèrement supérieur à la MAE, cela indique la présence de quelques erreurs de prédiction plus importantes (le RMSE pénalisant plus lourdement les grands écarts).

3. **La Capacité Prédictive ($R^2 \approx 28.4%$) :**
   * Un coefficient $R^2$ d'environ **28.4%** montre que près de la moitié de la variance des fluctuations de la santé mentale s'explique directement par les facteurs comportementaux mesurés (sommeil, activité physique, temps d'écran/IA).
   * Le reste de la variance (environ 71%) dépend de facteurs externes non présents dans ce dataset (génétique, situation familiale, stress académique pur, vie sociale réelle, etc.).

4. **La Surprise LightGBM :**
   * De manière inattendue, le modèle **LightGBM** obtient des résultats légèrement en deçà du Random Forest et de la régression Ridge linéaire. Cela peut s'expliquer par un besoin de réglage plus fin de ses hyperparamètres (comme la régularisation ou le taux d'apprentissage) pour éviter un léger surapprentissage sur ce dataset.

# 3.2 Tâche 2 : Classification (Étudiant à Risque)


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score,
                               f1_score, classification_report)

# =========================================================================
# 1. FONCTION D'ÉVALUATION POUR LA CLASSIFICATION BINAIRE
# =========================================================================
def evaluer_classification(model, X_train, y_train, X_test, y_test, nom):
    """
    Entraîne le modèle de classification et l'évalue sur le jeu de test.
    Calcule et affiche :
      - ROC-AUC : Évalue la capacité globale du modèle à séparer les deux classes.
      - PR-AUC : Plus adapté pour les classes déséquilibrées (focus sur la détection des minorités).
      - F1-Score : Moyenne harmonique entre la précision et le rappel.
    """
    # Entraînement du pipeline (preprocessing + modèle) sur l'ensemble d'entraînement
    model.fit(X_train, y_train)

    # Récupération des probabilités d'appartenance à la classe positive (étudiant à risque)
    y_proba = model.predict_proba(X_test)[:, 1]

    # Prédiction directe des classes (0 ou 1)
    y_pred  = model.predict(X_test)

    # Calcul des métriques d'évaluation clés
    roc_auc = roc_auc_score(y_test, y_proba)
    pr_auc  = average_precision_score(y_test, y_proba)
    f1      = f1_score(y_test, y_pred)

    # Affichage clair et structuré des métriques
    print(f"{nom:<30} ROC-AUC={roc_auc:.4f}  PR-AUC={pr_auc:.4f}  F1={f1:.4f}")

    # Retourne le modèle, les probabilités et les scores calculés pour archivage
    return model, y_proba, {'roc_auc': roc_auc, 'pr_auc': pr_auc, 'f1': f1}

print("=" * 70)
print("CLASSIFICATION — Détection des étudiants à risque (Mental_Health < 60)")
print("=" * 70)

# Initialisation des conteneurs pour stocker les résultats et les probabilités prédites
resultats_clf = {}
probas = {}

# =========================================================================
# 2. MODÈLE DE RÉFÉRENCE : DUMMY CLASSIFIER (Baseline)
# =========================================================================
# Ce modèle fait des prédictions aléatoires stratifiées. Tout vrai modèle doit faire mieux.
pipe_dummy = Pipeline([('prep', preprocesseur),
                        ('model', DummyClassifier(strategy='stratified', random_state=RANDOM_STATE))])
_, probas['Dummy'], resultats_clf['Dummy'] = evaluer_classification(
    pipe_dummy, X_train, y_train_clf, X_test, y_test_clf, 'Dummy (baseline)')

# =========================================================================
# 3. MODÈLE 1 : RÉGRESSION LOGISTIQUE
# =========================================================================
# Modèle linéaire classique.
# class_weight='balanced' rééquilibre automatiquement la pénalité pour les classes minoritaires.
pipe_lr = Pipeline([('prep', preprocesseur),
                     ('model', LogisticRegression(class_weight='balanced', max_iter=1000,
                                                   random_state=RANDOM_STATE))])
_, probas['LogisticRegression'], resultats_clf['LogisticRegression'] = evaluer_classification(
    pipe_lr, X_train, y_train_clf, X_test, y_test_clf, 'Logistic Regression')

# =========================================================================
# 4. MODÈLE 2 : RANDOM FOREST CLASSIFIER
# =========================================================================
# Forêt d'arbres de décision pour capturer des règles de décision non-linéaires complexes.
# class_weight='balanced' permet également d'ajuster les poids face au déséquilibre des classes.
pipe_rf_clf = Pipeline([('prep', preprocesseur),
                         ('model', RandomForestClassifier(n_estimators=200, max_depth=12,
                                                           class_weight='balanced',
                                                           n_jobs=-1, random_state=RANDOM_STATE))])
_, probas['RandomForest'], resultats_clf['RandomForest'] = evaluer_classification(
    pipe_rf_clf, X_train, y_train_clf, X_test, y_test_clf, 'Random Forest')

# =========================================================================
# 5. MODÈLE 3 : LIGHTGBM CLASSIFIER (Gradient Boosting)
# =========================================================================
# Modèle basé sur le boosting d'arbres décisionnels rapides et très performants.
# is_unbalance=True gère automatiquement le fort déséquilibre de notre variable cible (~10% de positifs).
pipe_lgbm_clf = Pipeline([('prep', preprocesseur),
                           ('model', lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05,
                                                         num_leaves=31, is_unbalance=True,
                                                         n_jobs=-1, random_state=RANDOM_STATE,
                                                         verbose=-1))])
_, probas['LightGBM'], resultats_clf['LightGBM'] = evaluer_classification(
    pipe_lgbm_clf, X_train, y_train_clf, X_test, y_test_clf, 'LightGBM')

### **3.2.1 Interprétation des Résultats de Classification**

Voici l'analyse comparative des performances de nos modèles pour identifier les étudiants dits **"à risque"** (score de santé mentale inférieur à 60) :

| Modèle | ROC-AUC | PR-AUC | F1-Score |
| :--- | :---: | :---: | :---: |
| **LightGBM** | 0.7801 | **0.3458** | **0.3866** |
| **Random Forest** | **0.7807** | 0.3137 | 0.3788 |
| **Logistic Regression** | 0.7783 | 0.3047 | 0.3157 |
| *Dummy (Baseline)* | 0.5033 | 0.1044 | 0.1091 |

#### **Observations et Analyses Clés :**

1. **La supériorité globale de LightGBM pour les classes déséquilibrées**
   * Bien que le **Random Forest** détienne le meilleur score **ROC-AUC (0.7807)**, c'est **LightGBM** qui se distingue comme le modèle le plus performant en situation réelle de déséquilibre de classe, affichant le meilleur **PR-AUC (0.3458)** et le meilleur **F1-Score (0.3866)**.
   * Rappel : les étudiants "à risque" ne représentent que **10,4%** du dataset. En cas de fort déséquilibre, le score ROC-AUC peut être flatteur et trompeur. C'est l'aire sous la courbe Précision-Rappel (**PR-AUC**) qui est la métrique la plus exigeante et robuste. LightGBM s'impose donc comme le meilleur choix pour cibler efficacement les étudiants en détresse.

2. **Apport des poids de classes (`class_weight` et `is_unbalance`)**
   * Tous nos modèles actifs surpassent largement la baseline aléatoire (*Dummy*).
   * L'utilisation d'options de pondération (`class_weight='balanced'` pour la Régression Logistique / Random Forest et `is_unbalance=True` pour LightGBM) a permis aux algorithmes de ne pas ignorer la classe minoritaire des étudiants à risque, augmentant drastiquement le Rappel (capacité à capter les vrais signaux de détresse).

3. **Limites de la classification brute**
   * Un F1-Score de **0.3866** montre que la tâche de classification pure reste difficile. Cela indique un arbitrage (compromis) entre la précision (éviter les fausses alertes) et le rappel (détecter un maximum d'étudiants en difficulté).
   * Dans un contexte de santé publique ou scolaire, on préférera souvent un modèle avec un **fort rappel** (quitte à avoir quelques faux positifs) pour s'assurer qu'aucun étudiant en souffrance réelle ne soit oublié par les systèmes de prévention.

# 4. Optimisation des Hyperparamètres avec Optuna


In [ ]:
# pip install optuna

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.model_selection import cross_val_score

def optimiser_lgbm_classification(X_train, y_train, n_trials=60):
    """Optimisation bayésienne de LightGBM avec Optuna."""
    def objective(trial):
        params = {
            'n_estimators'      : trial.suggest_int('n_estimators', 100, 800),
            'learning_rate'     : trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'num_leaves'        : trial.suggest_int('num_leaves', 20, 100),
            'min_child_samples' : trial.suggest_int('min_child_samples', 10, 80),
            'subsample'         : trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree'  : trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'reg_alpha'         : trial.suggest_float('reg_alpha', 1e-4, 5.0, log=True),
            'reg_lambda'        : trial.suggest_float('reg_lambda', 1e-4, 5.0, log=True),
            'is_unbalance'      : True,
            'random_state'      : RANDOM_STATE,
            'n_jobs'            : -1,
            'verbose'           : -1
        }
        # Initialisation du modèle LGBM
        model = lgb.LGBMClassifier(**params)

        # IMPORTANT : On associe le préprocesseur au modèle pour transformer le texte avant l'évaluation
        pipeline_opt = Pipeline([
            ('prep', preprocesseur),
            ('model', model)
        ])

        cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
        scores = cross_val_score(pipeline_opt, X_train, y_train, cv=cv,
                                  scoring='average_precision', n_jobs=-1)
        return scores.mean()

    study = optuna.create_study(direction='maximize',
                                  sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    print(f"\nMeilleur PR-AUC (CV) : {study.best_value:.4f}")
    print(f"Meilleurs hyperparamètres :")
    for k, v in study.best_params.items():
        print(f"  {k} : {v}")

    return study

# Lancer l'optimisation
study = optimiser_lgbm_classification(X_train, y_train_clf, n_trials=40)

# Modèle final optimisé
modele_final = Pipeline([
    ('prep', preprocesseur),
    ('model', lgb.LGBMClassifier(**study.best_params, is_unbalance=True,
                                   random_state=RANDOM_STATE, n_jobs=-1, verbose=-1))
])
modele_final.fit(X_train, y_train_clf)
y_proba_final = modele_final.predict_proba(X_test)[:, 1]

print(f"\nPerformance finale sur le test :")
print(f"  ROC-AUC : {roc_auc_score(y_test_clf, y_proba_final):.4f}")
print(f"  PR-AUC  : {average_precision_score(y_test_clf, y_proba_final):.4f}")

# =========================================================================
# 1. PRÉPARATION ET STRUCTURATION DES RÉSULTATS DES DEUX MODÈLES
# =========================================================================
# Comparaison des métriques clés entre le LightGBM par défaut et le LightGBM optimisé
comparaison_data = {
    "Modèle": ["LightGBM (Par défaut)", "LightGBM (Optimisé via Optuna)"],
    "ROC-AUC": [resultats_clf['LightGBM']['roc_auc'], roc_auc_score(y_test_clf, y_proba_final)],
    "PR-AUC": [resultats_clf['LightGBM']['pr_auc'], average_precision_score(y_test_clf, y_proba_final)],
    "F1-Score": [resultats_clf['LightGBM']['f1'], f1_score(y_test_clf, (y_proba_final >= 0.5).astype(int))]
}
df_comp = pd.DataFrame(comparaison_data)

# Application d'un style esthétique sur le tableau de comparaison
styled_comp = df_comp.style.background_gradient(cmap='YlGnBu', subset=['ROC-AUC', 'PR-AUC', 'F1-Score'])\
    .format({'ROC-AUC': '{:.4f}', 'PR-AUC': '{:.4f}', 'F1-Score': '{:.4f}'})\
    .set_properties(**{
        'text-align': 'center',
        'border': '1px solid #dee2e6',
        'font-family': 'sans-serif',
        'padding': '12px'
    })\
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center')]}
    ])

# =========================================================================
# 2. AFFICHAGE DE LA COMPARAISON ESTHÉTIQUE EN HTML
# =========================================================================
display(HTML("""
<div style='font-family: sans-serif; margin-bottom: 20px;'>
    <h2 style='color: #2e86c1; border-bottom: 3px solid #2e86c1; padding-bottom: 10px;'>🏆 Résultats de l'Optimisation des Hyperparamètres (Optuna)</h2>
    <p style='color: #566573; font-size: 15px;'>
        L'optimisation bayésienne a été menée sur 40 itérations en utilisant le <b>StratifiedKFold</b> à 3 plis pour maximiser l'aire sous la courbe Précision-Rappel (PR-AUC).
        Voici le comparatif des performances du modèle final optimisé sur l'ensemble de test face à sa version brute :
    </p>
</div>
"""))
display(styled_comp)

### **🔍 Analyse et Interprétation des Résultats de l'Optimisation**

#### **1. Analyse des Hyperparamètres Sélectionnés**
* **`learning_rate` (~0.062)** & **`n_estimators` (378)** : Optuna a identifié qu'un taux d'apprentissage modéré couplé à un nombre d'estimateurs intermédiaire est plus efficace qu'un grand nombre d'arbres à apprentissage trop rapide, évitant ainsi un surapprentissage rapide du gradient boosting.
* **`num_leaves` (95)** : Une structure d'arbre relativement complexe (jusqu'à 95 feuilles) a été choisie, ce qui permet de capturer les interactions combinatoires complexes et non-linéaires entre nos variables (comme l'effet simultané du manque de sommeil, du temps d'écran et du faible taux d'activité physique).
* **`colsample_bytree` (~0.63) & `subsample` (~0.68)** : Ces paramètres agissent comme des régulateurs en limitant la quantité de variables et d'échantillons examinés pour chaque arbre, améliorant fortement la robustesse du modèle face au bruit numérique.

#### **2. Évolution des Performances de Classification**
* **Aire sous la courbe Précision-Rappel (PR-AUC : 0.3458 → 0.3506)** :
  La métrique clé **PR-AUC** s'est améliorée sur le jeu de test non vu, passant de **34.58% à 35.06%**. Dans un contexte de fort déséquilibre de classes (environ 10.4% d'étudiants à risque), cette hausse montre que le modèle optimisé isole mieux les cas positifs réels des fausses alertes.
* **ROC-AUC (0.7801 → 0.7700)** :
  On observe une infime baisse sur la courbe ROC globale, mais cela est tout à fait acceptable. En effet, l'optimisation a spécifiquement ciblé la précision et le rappel (PR-AUC) plutôt que le ROC-AUC (qui a tendance à être trop optimiste en présence de classes déséquilibrées).

#### **3. Conclusion et Recommandations Métier**
Grâce à l'ajustement automatique d'Optuna, notre modèle **LightGBM optimisé** s'affirme comme le meilleur outil prédictif pour cibler les étudiants en détresse psychologique.

Dans un contexte opérationnel d'aide ou d'accompagnement scolaire :
1. Nous recommandons de fixer un seuil de décision personnalisé selon les ressources de l'école (par exemple, privilégier un rappel très élevé pour ne rater aucun profil d'étudiant réellement en souffrance).
2. Les variables telles que le **`Screen_Sleep_Ratio`** et la **`Heavy_Social_Media`** devront être au cœur des campagnes de prévention pour sensibiliser les jeunes sur l'importance du sommeil face au temps passé sur les écrans.

# 5. Interprétabilité avec SHAP

In [ ]:
import shap

# Récupérer le modèle entraîné et les données transformées
model_lgbm = modele_final.named_steps['model']
X_train_transformed = modele_final.named_steps['prep'].transform(X_train)
X_test_transformed  = modele_final.named_steps['prep'].transform(X_test)

# Noms des features après transformation
feature_names = (modele_final.named_steps['prep']
                 .get_feature_names_out())

# Calcul des valeurs SHAP
explainer = shap.TreeExplainer(model_lgbm)
shap_values = explainer.shap_values(X_test_transformed)

# Pour la classification binaire, prendre la classe positive
shap_pos = shap_values[1] if isinstance(shap_values, list) else shap_values

# Summary plot
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_pos, X_test_transformed,
                   feature_names=feature_names,
                   max_display=15, show=False)
plt.title('SHAP Summary Plot — Détection des étudiants à risque', fontsize=13)
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=120, bbox_inches='tight')
plt.show()

# Bar plot : importance globale
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_pos, X_test_transformed,
                   feature_names=feature_names,
                   plot_type='bar', max_display=15, show=False)
plt.title('SHAP Feature Importance — Impact moyen absolu', fontsize=13)
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=120, bbox_inches='tight')
plt.show()

### **🔍 Analyse et Interprétation des Résultats SHAP**

Les graphiques issus de l'analyse **SHAP (SHapley Additive exPlanations)** permettent de décoder la « boîte noire » du modèle LightGBM en quantifiant la contribution individuelle de chaque variable pour chaque prédiction d'étudiant « à risque ».

---

#### **1. Les Variables Clés Majeures (Top Facteurs)**

* **`Screen_Sleep_Ratio` (Le déséquilibre Écran/Sommeil)** :
  * **Observation** : C'est la variable la plus importante tout en haut du graphique.
  * **Interprétation** : Les points rouges (valeur élevée de ce ratio) sont fortement concentrés à droite (valeurs SHAP positives). Cela signifie qu'un étudiant qui a un temps d'exposition aux écrans disproportionné par rapport à sa durée de sommeil présente un risque drastiquement plus élevé de basculer en détresse psychologique.

* **`Daily_Social_Media_Hours` & `Total_Screen_Time` (Temps global d'exposition numérique)** :
  * **Observation** : Ces deux variables occupent les deuxième et troisième places de l'importance globale.
  * **Interprétation** : Sans surprise, une valeur élevée (rouge) pousse fortement la prédiction vers la droite (classe 1 : à risque). Le temps d'écran récréatif brut reste un indicateur direct très fort du niveau d'anxiété ou d'isolement.

* **`Sleep_Hours` & `Sleep_Deficit` (L'impact du sommeil)** :
  * **Observation** : Un grand nombre d'heures de sommeil (points bleus à droite pour le déficit ou rouges à gauche pour les heures de sommeil) a une valeur SHAP très négative.
  * **Interprétation** : Le sommeil est le principal **bouclier protecteur**. Plus l'étudiant dort (valeur élevée de `Sleep_Hours` ou faible de `Sleep_Deficit`), plus sa probabilité d'être classé « à risque » diminue.

---

#### **2. Les Variables Neutres ou Moins Influentes**

* **`Daily_AI_Tool_Usage_Hours` (L'utilisation de l'IA)** :
  * **Observation** : Son importance est nettement plus faible que celle des réseaux sociaux.
  * **Interprétation** : L'utilisation d'outils d'IA (ChatGPT, etc.) a un effet marginal sur la santé mentale. Contrairement aux réseaux sociaux, l'usage de l'IA est souvent à but productif/académique et ne semble pas induire de manière directe de détresse psychologique importante.

* **`Gender` & `Education_Level` (Données Démographiques)** :
  * **Observation** : Ces variables catégorielles se situent au bas du classement de l'importance absolue moyenne.
  * **Interprétation** : Le niveau d'éducation ou le genre de l'étudiant n'ont presque aucun poids prédictif direct sur la détérioration de la santé mentale. Le problème est purement **comportemental et lié à l'hygiène de vie**, transcendant les barrières de genre ou d'âge.

---

#### **3. Synthèse Métier pour les Décideurs**

Pour mener des actions de prévention ciblées au sein de l'établissement :
1. **La priorité absolue** n'est pas d'interdire simplement les réseaux sociaux, mais d'enseigner la **régulation du temps d'écran avant le coucher** pour rétablir un ratio sain entre le numérique et le repos corporel.
2. **Encourager l'activité physique** (`Activity_Ratio`) qui se confirme graphiquement comme un facteur modérateur réduisant la probabilité de risque.

# **6. Explication Individuelle (Waterfall Plot)**


In [ ]:
# Choisir un étudiant à risque élevé
idx_high_risk = np.argmax(y_proba_final)
print(f"Étudiant #{idx_high_risk}")
print(f"Probabilité prédite : {y_proba_final[idx_high_risk]:.4f}")
print(f"Vraie valeur : {y_test_clf.iloc[idx_high_risk]}")

# Caractéristiques
print(f"\nCaractéristiques :")
# print(X_test.iloc[idx_high_risk])
# Extraction des valeurs pour l'étudiant sélectionné

profil_etudiant = X_test.iloc[idx_high_risk].to_frame(name='Valeur')

# Ajout d'une colonne d'explication ou d'unité pour rendre le tableau plus parlant
commentaires = {
    'Age': "15 ans (Lycée)",
    'Gender': "Femme",
    'Education_Level': "Lycée (High School)",
    'Daily_Social_Media_Hours': "8.01 heures / jour (Utilisation intensive)",
    'Daily_AI_Tool_Usage_Hours': "2.94 heures / jour (Utilisation modérée)",
    'Sleep_Hours': "2.75 heures / nuit (Privation critique de sommeil)",
    'Physical_Activity_Hours': "0.00 heure / jour (Sédentarité absolue)",
    'Physical_Health_Score': "70.03 / 100 (Score dégradé)",
    'Total_Screen_Time': "10.95 heures / jour (Exposition très forte)",
    'Digital_Balance': "5.07 (Forte dominance des réseaux sociaux)",
    'Sleep_Deficit': "5.25 heures de sommeil manquantes / nuit",
    'Activity_Ratio': "0.00 (Aucune compensation par le sport)",
    'Screen_Sleep_Ratio': "3.84 (Ratio critique écran/sommeil)",
    'Heavy_Social_Media': "1 (Oui, > 6h de réseaux sociaux)",
    'Sleep_Deprived': "1 (Oui, < 6h de sommeil)"
}

profil_etudiant['Description / Impact'] = profil_etudiant.index.map(commentaires)

# Style du tableau (Correction : suppression de highlight_between sur types de données mixtes)
styled_profil = profil_etudiant.style.set_properties(**{
    'text-align': 'left',
    'border': '1px solid #dee2e6',
    'font-family': 'sans-serif',
    'padding': '10px'
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#e74c3c'), ('color', 'white'), ('font-weight', 'bold'), ('font-size', '14px')]}
])

display(HTML("""
<div style='font-family: sans-serif; margin-bottom: 15px;'>
    <h3 style='color: #c0392b; border-bottom: 2px solid #c0392b; padding-bottom: 8px;'>📋 Profil Détaillé des Caractéristiques : Étudiant #271</h3>
    <p style='color: #7f8c8d; font-size: 14px;'>Ce tableau regroupe les caractéristiques comportementales de l'étudiant ciblé par notre modèle comme étant à très haut risque (99.07%).</p>
</div>
"""))
display(styled_profil)


# Correction de l'erreur d'indexation sur expected_value
if isinstance(explainer.expected_value, (list, np.ndarray)) and len(explainer.expected_value) > 1:
    base_val = explainer.expected_value[1]
else:
    base_val = explainer.expected_value

# Waterfall plot
shap.waterfall_plot(
    shap.Explanation(values=shap_pos[idx_high_risk],
                      base_values=base_val,
                      data=X_test_transformed[idx_high_risk],
                      feature_names=feature_names),
    max_display=12, show=False
)
plt.title('Explication individuelle — Étudiant à risque élevé', fontsize=12)
plt.tight_layout()
plt.savefig('shap_waterfall.png', dpi=120, bbox_inches='tight')
plt.show()

### **💡 Analyse et Interprétation Métier de l'Explication Individuelle**

L'analyse de l'**Étudiant #271** (profil à très haut risque détecté à **99,07 %**) met en lumière la façon dont le modèle formule ses décisions face à des comportements extrêmes :

#### **1. Les Facteurs Majeurs de Risque accumulés**
* **Le déséquilibre Écran/Sommeil (`Screen_Sleep_Ratio = 3.84`)** : C'est le levier le plus déterminant pour pousser la prédiction vers la classe à risque. L'étudiant passe presque 4 fois plus de temps devant un écran (10,95 heures) que dans son lit (2,75 heures).
* **La privation critique de sommeil (`Sleep_Hours = 2.75` et `Sleep_Deficit = 5.25`)** : Le manque de sommeil est un inducteur de fatigue cognitive majeure, agissant directement sur l'humeur et la vulnérabilité psychologique.
* **La sédentarité absolue (`Physical_Activity_Hours = 0`)** : L'absence totale de sport élimine la seule barrière protectrice naturelle contre l'anxiété.

#### **2. Focus sur le cas du Faux Positif d'un point de vue de la Cible Réelle**
* **Vraie valeur : 0** (L'étudiant n'est pas classé à risque dans l'enquête réelle, son score étant supérieur à 60).
* **Analyse de la décision** : Bien que ce cas soit techniquement un **faux positif** pour l'algorithme, la décision reste hautement cohérente d'un point de vue de la prévention. Cet adolescent cumule tous les comportements précurseurs les plus alarmants du dataset. Le modèle a donc eu **parfaitement raison de lever une alerte maximale**, agissant comme un outil de détection précoce indispensable pour l'administration scolaire avant que la situation de l'étudiant ne s'aggrave concrètement.

# **7. Validation Croisée et Diagnostic**

### Valider la robustesse de notre modèle final de classification (**LightGBM**) et d'évaluer son comportement d'apprentissage


In [ ]:
from sklearn.model_selection import learning_curve

# 1. StratifiedKFold(n_splits=5, ...) :
# Sépare les données d'entraînement en 5 plis (folds) tout en conservant la proportion
# d'étudiants à risque (~10.4%) dans chaque pli. Cela évite le biais d'évaluation lié au fort déséquilibre des classes.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# 2. cross_val_score(..., scoring='average_precision') :
# Calcule le PR-AUC (aire sous la courbe Précision-Rappel) sur chaque pli. C'est la métrique la plus fiable
# pour juger la capacité du modèle à cibler les cas positifs réels sans générer de fausses alertes massives.
cv_scores = cross_val_score(modele_final, X_train, y_train_clf,
                             cv=cv, scoring='average_precision', n_jobs=-1)

print(f"PR-AUC en validation croisée (5-fold) :")
print(f"  Moyenne : {cv_scores.mean():.4f}")
print(f"  Écart-type : {cv_scores.std():.4f}")
print(f"  Scores : {cv_scores.round(4)}")

# 3. learning_curve(...) :
# Génère le score d'entraînement (Train) et de validation (Validation) pour différentes tailles d'échantillons
# afin de diagnostiquer le compromis Biais/Variance (sous-apprentissage ou surapprentissage).
train_sizes, train_scores, val_scores = learning_curve(
    modele_final, X_train, y_train_clf,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=3, scoring='average_precision', n_jobs=-1
)

plt.figure(figsize=(10, 5))
plt.plot(train_sizes, train_scores.mean(axis=1), 'o-', label='Train')
plt.plot(train_sizes, val_scores.mean(axis=1), 's-', label='Validation')
plt.fill_between(train_sizes,
                  train_scores.mean(axis=1) - train_scores.std(axis=1),
                  train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.2)
plt.fill_between(train_sizes,
                  val_scores.mean(axis=1) - val_scores.std(axis=1),
                  val_scores.mean(axis=1) + val_scores.std(axis=1), alpha=0.2)
plt.xlabel('Taille du training set')
plt.ylabel('PR-AUC')
plt.title('Courbe d\'apprentissage — LightGBM')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('learning_curve.png', dpi=120, bbox_inches='tight')
plt.show()

### **🔍 Analyse et Interprétation des Résultats de Validation Croisée**

* **PR-AUC Moyen (5-Fold) : 31.50 %** (avec un écart-type très faible de **0.66 %**).
* Les scores par pli sont extrêmement stables : `[0.3226, 0.3220, 0.3050, 0.3118, 0.3138]`.
* **Conclusion Métier** : La faible variance des scores confirme que les performances du modèle ne dépendent pas d'un découpage chanceux des données. Le modèle est **robuste et généralise de manière stable** sur de nouvelles données d'étudiants.

---

#### **3. Interprétation de la Courbe d'Apprentissage (*Learning Curve*)**
* **Courbe d'entraînement (Train - ligne bleue)** : Le score de précision moyenne reste constamment parfait ou très proche de 1.0 (100%). C'est une caractéristique typique des algorithmes de boosting (comme LightGBM) qui ajustent de manière très agressive les données d'apprentissage.
* **Courbe de validation (Validation - ligne orange)** : Le score de validation démarre aux alentours de **21.0 %** pour de petits volumes d'entraînement, puis progresse de manière constante à mesure que la taille du jeu d'entraînement augmente, pour se stabiliser à son maximum près de **31.5 % - 32.0 %**.
* **Diagnostic (Surapprentissage / Overfitting)** :
  * L'écart persistant (*gap*) entre la courbe d'entraînement et celle de validation indique un **léger surapprentissage**.
  * Cependant, la courbe de validation continue de croître légèrement ou se stabilise, montrant que l'apport de données supplémentaires a été bénéfique pour la généralisation du modèle.
  * Pour limiter cet écart dans une phase future, on pourrait renforcer la régularisation de LightGBM (ex: augmenter `reg_alpha` ou `reg_lambda`, ou restreindre davantage `num_leaves`).

## **🏁 8. Résumé Global du Rapport & Conclusions**

### **📊 Synthèse des Principales Découvertes (EDA)**
* **La crise du sommeil** : Les étudiants dorment en moyenne **6,55 heures** par nuit, un chiffre largement inférieur au seuil recommandé de 8h.
* **L'hégémonie des écrans** : Avec une moyenne de **4,54 heures par jour** passées sur les réseaux sociaux, l'exposition numérique récréative s'impose comme un facteur environnemental omniprésent.
* **La synergie fatale** : L'analyse statistique montre que l'impact sur la santé mentale n'est pas uniquement lié au temps d'écran cumulé, mais au **déséquilibre extrême** mesuré par le `Screen_Sleep_Ratio`. Un ratio élevé est le vecteur n°1 de dégradation psychologique ($r = -0.43$).

### **🤖 Performances des Modèles & Validation**
* **Tâche de Régression** : Le modèle **Random Forest** a démontré la meilleure capacité prédictive avec un **$R^2$ de 28,41 %** et une **erreur moyenne absolue (MAE) de 6,2 points**, ce qui est exceptionnel pour la prédiction de comportements humains intrinsèquement bruités.
* **Tâche de Classification & Robustesse** : Le classifieur **LightGBM optimisé via Optuna** s'est montré le plus performant pour isoler les profils à haut risque avec un **PR-AUC stable de 35,06 %**.
* **Garantie de Robustesse** : Notre validation croisée à 5 plis a confirmé l'extrême stabilité temporelle et structurelle du modèle avec un score moyen de **31,50 % ($\pm$ 0,66 %)**, prouvant ainsi l'absence de surapprentissage critique et une excellente généralisation opérationnelle.

### **🛡️ Recommandations Stratégiques et Métier**
1. **Alerte Précoce Réaliste** : Le cas d'étude de l'**Étudiant #271** (probabilité de risque de **99,07 %**) démontre que même face à un "faux positif" mathématique temporaire, l'outil cible parfaitement les profils en situation de détresse comportementale aigüe (sommeil de 2h45, sédentarité totale, écrans > 10h), justifiant pleinement une intervention préventive.
2. **Rétablir l'hygiène de vie** : Les campagnes scolaires doivent cibler prioritairement le sevrage des écrans avant le coucher plutôt que l'interdiction globale des technologies.
3. **Valoriser l'activité physique** : Le sport s'impose comme l'unique bouclier naturel à effet positif significatif sur le score de bien-être mental.

# **Conclusion Générale**
Ce projet démontre qu'il est possible de prédire efficacement le risque de problèmes de santé mentale chez les étudiants à partir de leurs habitudes numériques et de vie. Les résultats montrent que :
1. Les réseaux sociaux sont le premier facteur de risque (impact SHAP le plus élevé)
2. Le sommeil joue un rôle protecteur majeur
3. L'activité physique est un facteur protecteur secondaire
4. L'usage des outils IA a un impact négatif modéré

Le modèle LightGBM optimisé permet une détection précoce avec un PR-AUC de **0.35**, ce qui pourrait être utilisé par les établissements pour identifier les étudiants nécessitant un accompagnement.